In [21]:
# Import des bibliothèques

# Bibliothèques de mabipulation des données
import pandas as pd
import numpy as np

# Bibiliothèques de ML
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.neighbors import NearestNeighbors


In [22]:
df = pd.read_csv("../DF_FILMS_TMDB.csv")


In [23]:
df.head(1)

,tconst,original_country,poster_url,title,overview,release_date,genres,type,youtube_url,averageRating,numVotes
0,tt0111161,US,https://image.tmdb.org/t/p/w500/t30GjttOdb5At1...,Les Évadés,"En 1947, Andy Dufresne, un jeune banquier, est...",1994-09-23,Drama,Trailer,https://www.youtube.com/watch?v=UIzBz2hYnwc,9.3,3180532


In [24]:
df_trie = df[['original_country', 'title', 'release_date', 'genres', 'averageRating', 'numVotes']]
df_trie.head(20)

,original_country,title,release_date,genres,averageRating,numVotes
0,US,Les Évadés,1994-09-23,Drama,9.3,3180532
1,US,The Dark Knight : Le Chevalier noir,2008-07-16,"Action,Crime,Drama",9.1,3159860
2,"US, GB",Inception,2010-07-15,"Action,Adventure,Sci-Fi",8.8,2809674
3,US,Fight Club,1999-10-15,"Crime,Drama,Thriller",8.8,2599213
4,US,Interstellar,2014-11-05,"Adventure,Drama,Sci-Fi",8.7,2518486
5,US,Forrest Gump,1994-06-23,"Drama,Romance",8.8,2490710
6,US,Pulp Fiction,1994-09-10,"Crime,Drama",8.8,2430214
7,US,Matrix,1999-03-31,"Action,Sci-Fi",8.7,2242319
8,US,Le Seigneur des anneaux : La Communauté de l'a...,2001-12-18,"Adventure,Drama,Fantasy",8.9,2202083
9,US,Le Seigneur des anneaux : Le Retour du roi,2003-12-17,"Adventure,Drama,Fantasy",9.0,2160270


In [25]:
df_trie['original_country'].value_counts().head(30)

original_country
US                1457
JP                 180
FR                 169
US, GB              73
GB, US              28
CA, US              26
AU, US              12
BE, FR              11
CA, US, GB           4
CN, US               4
FR, GB               4
CA, FR               4
IE, US, GB           3
IE, US               3
FR, IT               3
AU, US, GB           2
CA, GB, US           2
NZ, US               2
DE, US               2
CA, DE, US           2
FR, ES               2
BE, FR, RO           2
DE, US, GB           1
US, CZ               1
DE, ES, GB, US       1
US, DE               1
US, IE, GB           1
US, ES               1
GB, IS, US           1
DK, US               1
Name: count, dtype: int64

In [ ]:
# J'encode tous mes genres avec get dummies, je l'utilise car il y a plus de 2 éléments a encoder

dummies = df_trie['genres'].str.get_dummies(',')
display(dummies)

,Action,Adventure,Animation,Biography,Comedy,Crime,Drama,Family,Fantasy,History,Horror,Music,Musical,Mystery,Romance,Sci-Fi,Sport,Thriller,War,Western
0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
1,1,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0
2,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
3,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,1,0,0
4,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2073,1,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2074,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0
2075,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
2076,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0


In [ ]:
# Je concatene mon df_trie avec le dummies pour ajouter les colonnes encodées

df_concat = pd.concat([df_trie, dummies], axis=1)
df_concat.head()

,original_country,title,release_date,genres,averageRating,numVotes,Action,Adventure,Animation,Biography,...,Horror,Music,Musical,Mystery,Romance,Sci-Fi,Sport,Thriller,War,Western
0,US,Les Évadés,1994-09-23,Drama,9.3,3180532,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,US,The Dark Knight : Le Chevalier noir,2008-07-16,"Action,Crime,Drama",9.1,3159860,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,"US, GB",Inception,2010-07-15,"Action,Adventure,Sci-Fi",8.8,2809674,1,1,0,0,...,0,0,0,0,0,1,0,0,0,0
3,US,Fight Club,1999-10-15,"Crime,Drama,Thriller",8.8,2599213,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
4,US,Interstellar,2014-11-05,"Adventure,Drama,Sci-Fi",8.7,2518486,0,1,0,0,...,0,0,0,0,0,1,0,0,0,0


In [ ]:
# Je crée une fonction pour regrouper tous les origines des pays en 3 sous groupes pour ensuite pouvoir utiliser cette colonne pour le ML

def categoriser(x):
    pays = str(x)
    if 'FR' in pays:
        return "FR"
    elif 'JP' in pays:
        return "JP"
    elif "US" in pays:
        return 'US'


# J'applique ma fonction a toute la colonne original country et dans ma nouvelle colonne j'aurais soit FR soit US soit JP

df_concat['pays_categorie'] = df_concat['original_country'].apply(categoriser)

In [29]:
display(df_concat['pays_categorie'].value_counts())

pays_categorie
US    1658
FR     238
JP     182
Name: count, dtype: int64

In [ ]:
# J'encode mes 3 catégories avec map et j'utilise 0, 1 et 2

df_concat['pays_categorie'] = df_concat['pays_categorie'].map({"US" : 0, "FR" : 1, "JP" : 2})


In [31]:
df_concat.head()

,original_country,title,release_date,genres,averageRating,numVotes,Action,Adventure,Animation,Biography,...,Music,Musical,Mystery,Romance,Sci-Fi,Sport,Thriller,War,Western,pays_categorie
0,US,Les Évadés,1994-09-23,Drama,9.3,3180532,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,US,The Dark Knight : Le Chevalier noir,2008-07-16,"Action,Crime,Drama",9.1,3159860,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,"US, GB",Inception,2010-07-15,"Action,Adventure,Sci-Fi",8.8,2809674,1,1,0,0,...,0,0,0,0,1,0,0,0,0,0
3,US,Fight Club,1999-10-15,"Crime,Drama,Thriller",8.8,2599213,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
4,US,Interstellar,2014-11-05,"Adventure,Drama,Sci-Fi",8.7,2518486,0,1,0,0,...,0,0,0,0,1,0,0,0,0,0


In [ ]:
# Je renomme mon df
df_encode = df_concat

In [ ]:
# Je le sauvegarde en csv
df_encode.to_csv("DF_ENCODE.csv")

In [34]:
# J'extrais l'année et le mois de ma colonne date

df_encode['année_sortie'] = pd.to_datetime(df_encode['release_date']).dt.year
df_encode['mois_sortie'] = pd.to_datetime(df_encode['release_date']).dt.month

In [35]:
df_encode.head(1)

,original_country,title,release_date,genres,averageRating,numVotes,Action,Adventure,Animation,Biography,...,Mystery,Romance,Sci-Fi,Sport,Thriller,War,Western,pays_categorie,année_sortie,mois_sortie
0,US,Les Évadés,1994-09-23,Drama,9.3,3180532,0,0,0,0,...,0,0,0,0,0,0,0,0,1994,9


In [36]:
# Ici je choisis d'utiliser le MinMaxScaler qui va mettre toutes les valeurs entre 0 et 1

scaler = MinMaxScaler()

# Je choisis mes colonnes numériques a scaler
cols_scaler = ['averageRating', 'numVotes', 'année_sortie', 'mois_sortie']

# Je mets a l'echelle mes colonnes numériques pour qu'aucune ne prenne le dessus sur les autres
df_encode[cols_scaler] = scaler.fit_transform(df_encode[cols_scaler])

In [37]:
df_encode.head()

,original_country,title,release_date,genres,averageRating,numVotes,Action,Adventure,Animation,Biography,...,Mystery,Romance,Sci-Fi,Sport,Thriller,War,Western,pays_categorie,année_sortie,mois_sortie
0,US,Les Évadés,1994-09-23,Drama,1.000000,1.000000,0,0,0,0,...,0,0,0,0,0,0,0,0,0.111111,0.727273
1,US,The Dark Knight : Le Chevalier noir,2008-07-16,"Action,Crime,Drama",0.913043,0.993490,1,0,0,0,...,0,0,0,0,0,0,0,0,0.500000,0.545455
2,"US, GB",Inception,2010-07-15,"Action,Adventure,Sci-Fi",0.782609,0.883214,1,1,0,0,...,0,0,1,0,0,0,0,0,0.555556,0.545455
3,US,Fight Club,1999-10-15,"Crime,Drama,Thriller",0.782609,0.816938,0,0,0,0,...,0,0,0,0,1,0,0,0,0.250000,0.818182
4,US,Interstellar,2014-11-05,"Adventure,Drama,Sci-Fi",0.739130,0.791516,0,1,0,0,...,0,0,1,0,0,0,0,0,0.666667,0.909091


In [38]:
# Je définis mes features X

X = df_encode.drop(columns = ['original_country', 'title', 'release_date', 'genres'])

In [74]:
# Je définis mon modèle

model = NearestNeighbors(n_neighbors=6)  # On met 6 pour récupérer 6 films auxquels nous enleveront le film de l'input

# et je l'entraine sur les autres films qui sont les données à recommander.

model.fit(X)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",6
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [68]:
# Recherche des 5 films à recommander

distance, indices = model.kneighbors(X)

In [69]:
print(distance) # Retourne la distance entre le film et ses plus proches voisins

[[0.         0.75309369 0.84194994 1.01559866 1.01998123 1.05027987]
 [0.         0.49070644 0.61384497 1.00478892 1.0481562  1.0599592 ]
 [0.         0.59399305 0.60021507 0.63290814 0.69322595 0.71948898]
 ...
 [0.         0.12592948 0.1376537  0.18182008 0.19503324 0.20189598]
 [0.         0.1850317  0.2321797  0.28762956 0.29449254 0.37638788]
 [0.         0.11192668 0.20847269 0.28342964 0.32549996 0.33774213]]


In [70]:
print(indices) # Retournes les indexs des voisins

[[   0   40   71  106  151    6]
 [   1   11   18  119   87  233]
 [   2   28   33   51   48   39]
 ...
 [2075 1529 1517 1936 1813 1306]
 [2076  825 1207 1623 2048 1045]
 [2077  653  843  711 1392  470]]


In [71]:
indices.shape

(2078, 6)

In [54]:
# Recherche des films à recommander pour Inception

input_nom = "Inception"
display(df_encode[df_encode['title'] == input_nom ])

reco_indices = indices[0,]   # Je selectionne l'indice 0 de >> indices << (donc la premiere ligne en entier)
print("\n Les indices des 5 films recommandés sont: ",reco_indices)

,original_country,title,release_date,genres,averageRating,numVotes,Action,Adventure,Animation,Biography,...,Mystery,Romance,Sci-Fi,Sport,Thriller,War,Western,pays_categorie,année_sortie,mois_sortie
2,"US, GB",Inception,2010-07-15,"Action,Adventure,Sci-Fi",0.782609,0.883214,1,1,0,0,...,0,0,1,0,0,0,0,0,0.555556,0.545455



 Les indices des 5 films recommandés sont:  [  0  40  71 106 151]


In [55]:
# J'affiche maintenant les noms des films en fonctions des index

recommandations = df_encode.iloc[reco_indices,:10]
display(recommandations)

,original_country,title,release_date,genres,averageRating,numVotes,Action,Adventure,Animation,Biography
0,US,Les Évadés,1994-09-23,Drama,1.000000,1.000000,0,0,0,0
40,US,American Beauty,1999-09-15,Drama,0.565217,0.400981,0,0,0,0
71,US,Requiem for a Dream,2000-10-06,Drama,0.565217,0.304442,0,0,0,0
106,US,Gran Torino,2008-12-12,Drama,0.478261,0.269511,0,0,0,0
151,US,There Will Be Blood,2007-12-26,Drama,0.521739,0.220996,0,0,0,0


In [ ]:
# Création d'une fonction pour notre système de recommandation

def film_reco(film:str):
    info_film = df_encode[df_encode['title'] == film]
    indice_film = info_film.index[0]
    reco_indices = indices[indice_film][1:] # On commence à 1 pour exclure le film qu'on demande (il serait automatiquement recommandé dans le cas inverse)
    df_reco = df_encode.iloc[reco_indices,:4]
    return df_reco


In [ ]:
# J'utilise ma fonction sur plusieurs films

film_reco("Usual Suspects")

,original_country,title,release_date,genres
178,US,L.A. Confidential,1997-09-19,"Crime,Drama,Mystery"
10,US,Seven,1995-09-22,"Crime,Drama,Mystery"
245,US,Mystic River,2003-10-08,"Crime,Drama,Mystery"
512,US,Peur primale,1996-04-03,"Crime,Drama,Mystery"
82,US,Prisoners,2013-09-19,"Crime,Drama,Mystery"
